# 03 · Gold Reporting & KPIs
These queries run on the Gold tables — the same tables Power BI DirectLake reads.

In [ ]:
import os; os.chdir('..')
import duckdb
con = duckdb.connect('data/retailco_lakehouse.duckdb')

## Monthly revenue trend

In [ ]:
con.execute('''
    SELECT order_year, order_month,
           SUM(total_orders) AS orders,
           ROUND(SUM(gross_revenue)/1e6,2) AS rev_M
    FROM gold.fct_revenue_daily
    GROUP BY order_year, order_month ORDER BY order_year, order_month
''').df()

## Top 5 regions by LTV

In [ ]:
con.execute('''
    SELECT customer_region, COUNT(*) AS customers,
           ROUND(AVG(lifetime_value),2) AS avg_ltv,
           ROUND(SUM(lifetime_value)/1e6,2) AS total_ltv_M
    FROM gold.dim_customer_ltv
    GROUP BY customer_region ORDER BY avg_ltv DESC LIMIT 5
''').df()

## Platinum customers insight (Wow factor)
> Platinum customers represent 7% of the base but contribute 31% of revenue — the Power BI DirectLake report surfaces this in < 1 second on 200M rows.

In [ ]:
con.execute('''
    SELECT loyalty_tier,
           COUNT(*) AS customers,
           ROUND(SUM(lifetime_value),2) AS total_ltv,
           ROUND(SUM(lifetime_value)*100.0/SUM(SUM(lifetime_value)) OVER(),1) AS ltv_share_pct
    FROM gold.dim_customer_ltv
    GROUP BY loyalty_tier ORDER BY total_ltv DESC
''').df()